In [1]:
import cv2
import os
import pickle
from matplotlib import pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import numpy as np

In [2]:
image_dir = '/mnt/data/UserData/shichen/data/mscoco/val2014/'
save_dir = '/mnt/data/UserData/shichen/fmri_prototype/log_prototype_coco_4_4_1000_128/top_image_visualization_clipscore'
prototype_data = pickle.load(open('/mnt/data/UserData/shichen/fmri_prototype/log_prototype_coco_4_4_1000_128/topk_images_clipscore.pkl', 'rb'))

In [ ]:
# visualize raw image first
os.makedirs(os.path.join(save_dir, 'raw'), exist_ok=True)

def plot_image_grid(image_names, raw_dir, save_name="grid.png", target_size=(256, 256)):
    """
    Reads 20 images and saves a 5x4 grid plot.

    Parameters
    ----------
    image_names : list
        List of 20 image filenames (e.g., ["img1.jpg", ...])
    raw_dir : str
        Directory containing the raw .jpg images
    save_dir : str
        Directory to save the output plot
    save_name : str
        Name of the saved plot file
    """

    assert len(image_names) == 20, "You must provide exactly 20 image names."

    # Create figure (5 rows × 4 columns)
    fig, axes = plt.subplots(5, 4, figsize=(12, 15))

    for i, ax in enumerate(axes.flatten()):
        img_path = os.path.join(raw_dir, image_names[i]+'.jpg')
        img = Image.open(img_path).convert("RGB")
        img = img.resize(target_size, Image.Resampling.LANCZOS)

        ax.imshow(img)
        ax.axis("off")  # no ticks, no border, no title

    plt.tight_layout(pad=0)

    # Save the plot
    plt.savefig(save_name, bbox_inches="tight", pad_inches=0)
    plt.close()

for proto_id in range(len(prototype_data['image_name'])):
    plot_image_grid(prototype_data['image_name'][proto_id].tolist(), image_dir, os.path.join(save_dir, 'raw', str(proto_id)+'.jpg'))



In [13]:
# now visualize the attended image 
os.makedirs(os.path.join(save_dir, 'attention'), exist_ok=True)

def plot_overlay_grid_resized(
    image_names,
    heatmaps,
    raw_dir,
    save_name="overlay_grid.png",
    target_size=(256, 256),
    alpha=0.5,
    cmap="jet"
):
    """
    Resize images and heatmaps to target_size,
    overlay heatmaps, and save a 5x4 grid.

    Parameters
    ----------
    image_names : list of str (length 20)
    heatmaps : list of np.ndarray (length 20)
    raw_dir : str
    save_dir : str
    save_name : str
    target_size : tuple (width, height)
    alpha : float
    cmap : str
    """

    assert len(image_names) == 20, "Must provide 20 image names"
    assert len(heatmaps) == 20, "Must provide 20 heatmaps"

    fig, axes = plt.subplots(5, 4, figsize=(12, 15))

    for i, ax in enumerate(axes.flatten()):

        # ---- Load & Resize Image ----
        img_path = os.path.join(raw_dir, image_names[i]+'.jpg')
        img = Image.open(img_path).convert("RGB")
        img = img.resize(target_size, Image.Resampling.LANCZOS)
        img_np = np.array(img)

        # ---- Prepare & Resize Heatmap ----
        heatmap = heatmaps[i].astype(np.float32)

        # Normalize heatmap
        heatmap -= heatmap.min()
        if heatmap.max() > 0:
            heatmap /= heatmap.max()

        heatmap_resized = cv2.resize(
            heatmap,
            target_size,
            interpolation=cv2.INTER_LINEAR
        )

        # ---- Overlay ----
        ax.imshow(img_np)
        ax.imshow(heatmap_resized, cmap=cmap, alpha=alpha)
        ax.axis("off")

    plt.tight_layout(pad=0)

    plt.savefig(save_name, bbox_inches="tight", pad_inches=0)
    plt.close()
    
for proto_id in range(len(prototype_data['image_name'])):
    cur_attention = prototype_data['attention_map'][proto_id]
    plot_overlay_grid_resized(prototype_data['image_name'][proto_id].tolist(), cur_attention, image_dir, os.path.join(save_dir, 'attention', str(proto_id)+'.jpg'))